In [1]:
import re
from collections import Counter

# Step 1 & 2: Load corpus and build vocabulary with word frequencies
def build_vocabulary(corpus_text: str) -> dict:
    """Extracts words and builds a word-frequency dictionary."""
    words = re.findall(r'\w+', corpus_text.lower())
    return Counter(words)

# Step 4: Tokenize user search query
def tokenize_query(query: str) -> list[str]:
    """Cleans and splits a search query into tokens."""
    return re.findall(r'\w+', query.lower())

# Step 5: Identify out-of-vocabulary (OOV) words
def get_unknown_words(tokens: list[str], vocabulary: dict) -> list[str]:
    """Returns tokens that are not present in the vocabulary."""
    return [token for token in tokens if token not in vocabulary]

# Step 6: Calculate Levenshtein Edit Distance
def edit_distance(str1: str, str2: str) -> int:
    """Computes the minimum edit distance between two strings."""
    m, n = len(str1), len(str2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
        
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i - 1] == str2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j],      # Deletion
                                   dp[i][j - 1],      # Insertion
                                   dp[i - 1][j - 1])  # Substitution
    return dp[m][n]

# Step 7: Select the closest matching word
def find_closest_word(word: str, vocabulary: dict, max_distance: int = 2) -> str:
    """Finds the candidate with the lowest edit distance and highest frequency."""
    candidates = []
    
    # Optimization: Only evaluate words starting with same letter if vocabulary is large
    vocabulary_candidates = [w for w in vocabulary if abs(len(w) - len(word)) <= max_distance]
    
    for candidate in vocabulary_candidates:
        dist = edit_distance(word, candidate)
        if dist <= max_distance:
            # Sort priority: (lowest distance, highest word frequency)
            candidates.append((dist, -vocabulary[candidate], candidate))
            
    if not candidates:
        return word  # Fallback to original word if no match within max_distance
        
    candidates.sort()
    return candidates[0][2]

# Step 3, 8 & Pipeline Assembly: Process entire query
def correct_search_query(query: str, vocabulary: dict) -> str:
    """Runs the full correction pipeline on an input query."""
    tokens = tokenize_query(query)
    corrected_tokens = []
    
    for token in tokens:
        if token in vocabulary:
            corrected_tokens.append(token)
        else:
            corrected = find_closest_word(token, vocabulary)
            corrected_tokens.append(corrected)
            
    return " ".join(corrected_tokens)


# ==========================================
# Step 9: Testing the Complete System
# ==========================================
if __name__ == "__main__":
    # Sample corpus simulating standard search vocabulary
    sample_corpus = """
    machine learning artificial intelligence natural language processing 
    search engine optimization python algorithm data science database
    deep learning computer vision neural network model training
    """
    
    # Step 1 & 2: Build vocabulary
    vocab = build_vocabulary(sample_corpus)
    
    # Step 9: Test cases with single and multiple spelling errors
    test_queries = [
        "machne lernin",                    # Multiple typos
        "artifcial intelgence model",       # Multiple typos
        "pythn data scinece",               # Multiple typos
        "search engin optmization"          # Three typos
    ]
    
    print("--- SEARCH QUERY CORRECTION RESULTS ---\n")
    for original in test_queries:
        corrected = correct_search_query(original, vocab)
        print(f"Original Input : '{original}'")
        print(f"Corrected Query: '{corrected}'\n")

--- SEARCH QUERY CORRECTION RESULTS ---

Original Input : 'machne lernin'
Corrected Query: 'machine learning'

Original Input : 'artifcial intelgence model'
Corrected Query: 'artificial intelligence model'

Original Input : 'pythn data scinece'
Corrected Query: 'python data science'

Original Input : 'search engin optmization'
Corrected Query: 'search engine optimization'

